# BAMv3 실험용 데이터셋 생성

- 클린 IQ 신호 저장
- Training 시 on-the-fly로 노이즈 추가


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os, sys

# 상위 디렉토리의 utils 모듈 사용
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from utils.LoRa import LoRa
import importlib
import utils.my_lora_utils
importlib.reload(utils.my_lora_utils)
from utils.my_lora_utils import *

print("Loaded my_lora_utils from:", utils.my_lora_utils.__file__)


In [ ]:
# LoRa 파라미터 설정
sf = 9                      # Spreading Factor
bw = 250_000                # 250 kHz
OSF = 4                     # Oversampling Factor
fs = int(bw * OSF)          # 1 MHz

symbol_time = 2**sf / bw    # 한 심볼 시간
N = 2**sf                   # 가능한 심볼 개수 (0 ~ 511)

print(f"SF = {sf}")
print(f"BW = {bw/1e3:.1f} kHz")
print(f"fs = {fs/1e6:.3f} MHz (OSF = {fs/bw:.1f})")
print(f"Symbol time = {symbol_time*1e3:.3f} ms")

# LoRa 심볼 생성 클래스 초기화
lora = LoRa(sf, bw)


In [ ]:
# 데이터 저장 폴더 설정
base_dir = "dataset_v3_sf9_bw250k"
iq_dir = os.path.join(base_dir, "clean_iq")

os.makedirs(iq_dir, exist_ok=True)

print("Created/Checked folder:")
print(" -", iq_dir)


In [ ]:
# 모든 심볼(0 ~ N-1)에 대해 clean IQ 저장
GENERATE = False  # 실행 방지

if GENERATE:
    num_symbols = N  # 512
    print(f"Generating clean dataset for {num_symbols} symbols...")
    
    for sym in range(num_symbols):
        # 클린 심볼 IQ 생성
        x_clean = lora.gen_symbol_fs(sym, sf=sf, bw=bw, Fs=fs)
        
        # 클린 IQ 저장
        iq_path = os.path.join(iq_dir, f"sym_{sym:03d}_iq.npy")
        np.save(iq_path, x_clean.astype(np.complex64))
        
        if sym % 50 == 0:
            print(f"  - Generated symbol {sym}/{num_symbols-1}")
    
    print("✅ Clean dataset generation done.")
else:
    print("Dataset generation skipped (GENERATE=False)")
